In [3]:
library(tidyverse)
library(ggplot2)
library(data.table)
library(dtplyr)
library(arrow)

In [ ]:
files <- list.files(path = "D:/BNC Full Data/12-9_10AM Run/CSV",
                    pattern = "\\.csv$",
                    full.names = TRUE)

df_combined <- read_csv(files, id = "file_name")

write_rds(df_combined, "D:/BNC Full Data, 12-10 8PM Run/12-10 8PM Unprocessed Data.rds")
write_parquet(df_combined, "Results 12-10 8PM/12-10 8PM Unprocessed Data.parquet")

Rows: 106513970 Columns: 44
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (21): Sentence_ID, Filename, Modality, Sentence_Text, Word_Token, Phrase...
dbl (18): Sent_Verb_Count, Sent_Auxiliary_Count, Sent_Subject_Count, Sent_To...
lgl  (4): Sent_Transitive, Is_NP, NP_Is_Bare_NP, Is_Head_Noun

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Warning message in saveRDS(x, con, version = version, refhook = refhook, ascii = text):
"cannot open file 'D:/BNC Full Data, 12-10 8PM Run/12-10 8PM Unprocessed Data.rds': No such file or directory"


ERROR: Error in saveRDS(x, con, version = version, refhook = refhook, ascii = text): cannot open the connection


: 

In [ ]:
summary(df_combined)

  file_name         Sentence_ID          Filename           Modality        
 Length:106045470   Length:106045470   Length:106045470   Length:106045470  
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
                                                                            
                                                                            
                                                                            
                                                                            
 Sentence_Text      Sent_Verb_Count Sent_Auxiliary_Count Sent_Subject_Count
 Length:106045470   Min.   : 0.00   Min.   : 0.000       Min.   : 0.000    
 Class :character   1st Qu.: 2.00   1st Qu.: 1.000       1st Qu.: 1.000    
 Mode  :character   Median : 3.00   Median : 1.000       Median : 2.000    
                    Mean   : 3.07   Mean   : 1.662       Mean   : 1.917    
    

In [ ]:
# Creating Tags for first element in each NP

df_processed <- df_combined %>% 
    arrange(Sentence_ID, Word_Token_Index) %>% 
    group_by(Sentence_ID) %>% 
    mutate(
        prev_is_NP = lag(Is_NP, default = FALSE),
        prev_NP_Head_Text = lag(NP_Head_Text),

        first_token_of_NP = Is_NP & (!prev_is_NP | NP_Head_Text != prev_NP_Head_Text)
    ) %>% 
    ungroup() %>% 
    select(-prev_is_NP, -prev_NP_Head_Text) %>% 
#Propogates index of first NP down to the full phrase
    group_by(Sentence_ID, np_id = consecutive_id(Phrase_Token)) %>% 
    mutate(
        np_start_idx = ifelse(
            is.na (Phrase_Token) | Is_NP == FALSE, 
            NA,
            min(Word_Token_Index)
        )
    ) %>% 
    ungroup() %>% 
    select(-np_id) %>% 
# Creates within_file IDS and Within Chunk Ids
    mutate(
        within_file_id = str_extract(Sentence_ID, "(?<=_)\\d+") %>% 
        as.integer
    ) 

# Remove duplicate sentences:

df_processed <- df_processed %>% 
    arrange(Sentence_ID) %>% 
        group_by(Sentence_Text) %>%
        mutate(first_Sentence_ID = first(Sentence_ID)) %>%
        filter(Sentence_ID == first_Sentence_ID) %>%
        ungroup() %>%
        select(-first_Sentence_ID)

In [ ]:
summary(df_processed)

  file_name         Sentence_ID          Filename           Modality        
 Length:103426691   Length:103426691   Length:103426691   Length:103426691  
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
                                                                            
                                                                            
                                                                            
                                                                            
 Sentence_Text      Sent_Verb_Count  Sent_Auxiliary_Count Sent_Subject_Count
 Length:103426691   Min.   : 0.000   Min.   : 0.00        Min.   : 0.000    
 Class :character   1st Qu.: 2.000   1st Qu.: 1.00        1st Qu.: 1.000    
 Mode  :character   Median : 3.000   Median : 1.00        Median : 2.000    
                    Mean   : 3.072   Mean   : 1.66        Mean   : 1.919    

In [ ]:
write_rds(df_processed, "D:/BNC Full Data, 12-10 8PM Run/12-10 8PM Processed Data.rds")
write_parquet(df_processed, "Results 12-10 8PM/12-10 8PM Processed Data.parquet")

In [ ]:
df_sentences <- df_processed %>% 
    filter(
        Modality == "written", 
        Sent_Verb_Count == 1,
        Sent_Auxiliary_Count == 0,
        Sent_Subject_Count == 1,
        Sent_Tot_Obj_Count %in% 1,
        Sent_Dir_Object_Count == 1 ,
        Sent_Ind_Object_Count == 0,
        Sent_Sub_Conj_Count == 0,
        Sent_Coord_Conj_Count == 0, 
        Clausal_Complement_Count == 0,
        Sent_Relative_Clause_Count == 0, 
        Sent_Adv_Clause_Count == 0, 
        Sent_Prep_Phrase_Count == 0,
        Sent_Comma_Count == 0,
        !str_detect(Sentence_Text, "\\?"),
        Sent_Transitive == TRUE,
    )

df_sentences <- df_sentences %>% 
    mutate(
        definiteness = factor(NP_Definiteness,
        levels = c("indefinite", "definite"),
        labels = c("indef", "def"))
    ) %>% 
    mutate(
        argPos = factor(
            NP_Argument,
            levels = c("dir_object", "subject"),
            labels = c("obj", "sbj")
        )
    ) %>% 
    mutate(surprisal = Phrase_Surprisal) %>% 
    select(Sentence_ID, Sentence_Text, Phrase_Token, Word_Token, Word_Token_Index, Word_Surprisal, surprisal, definiteness, argPos, np_start_idx, within_file_id, within_chunk_id, Is_NP, Is_Head_Noun)

summary(df_sentences)

 Sentence_ID        Sentence_Text      Phrase_Token        Word_Token       
 Length:268332      Length:268332      Length:268332      Length:268332     
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
                                                                            
                                                                            
                                                                            
                                                                            
 Word_Token_Index Word_Surprisal     surprisal      definiteness  
 Min.   : 0.000   Min.   : 0.014   Min.   : 0.014   indef: 41354  
 1st Qu.: 1.000   1st Qu.:13.359   1st Qu.:13.831   def  :101264  
 Median : 3.000   Median :15.695   Median :15.828   NA's :125714  
 Mean   : 2.892   Mean   :16.403   Mean   :16.402                 
 3rd Qu.: 4.000   3rd Qu.:18.828   3rd Qu.:18.365

In [ ]:

write_rds(df_sentences, "D:/BNC Full Data, 12-10 8PM Run/12-10 8PM Filtered All Tokens Data.rds")
write_parquet(df_sentences, "Results 12-10 8PM/12-10 8PM All Tokens Data.parquet")

In [ ]:
df_nps <- df_sentences %>% 
    filter(
        Is_NP == TRUE,
        Is_Head_Noun == TRUE,
        definiteness %in% c("def", "indef"),
        argPos %in% c("obj", "sbj"),
        !is.na(surprisal),
    ) %>% 
    group_by(Sentence_ID) %>% # Drops sentences without one subject and one object
        filter(n() == 2 & n_distinct(argPos) == 2) %>%
        ungroup()

In [ ]:
write_rds(df_nps, "D:/BNC Full Data, 12-10 8PM Run/12-10 8PM NP only.rds")
write_parquet(df_sentences, "Results 12-10 8PM/12-10 8PM NP Only.parquet")

### Analyses and Plots

In [ ]:

# library(ggeffects)
    plotFont <- function(fontBase) { # Easy way to adjust font size for plots
        theme( # Add as a final ggplot object (no parentheses)
        plot.title = element_text(size = 14*fontBase),      # Title font size
        axis.title.x = element_text(size = 12*fontBase),    # X-axis title font size
        axis.title.y = element_text(size = 12*fontBase),    # Y-axis title font size
        axis.text.x = element_text(size = 10*fontBase),     # X-axis tick labels font size
        axis.text.y = element_text(size = 10*fontBase),     # Y-axis tick labels font size
        legend.text = element_text(size = 10*fontBase),     # Legend text size
        legend.title = element_text(size = 10*fontBase),    # Legend title size
        strip.text = element_text(size = 10*fontBase)
        )
    }

In [ ]:
count_table <- df_nps %>% 
    count(argPos, definiteness)
count_table

argPos,definiteness,n
<fct>,<fct>,<int>
obj,indef,899
obj,def,794
sbj,indef,383
sbj,def,1310


In [ ]:
mod_NP_Index <- lm(data=df_nps, surprisal ~ definiteness * argPos * np_start_idx)

summary(mod_NP_Index)


Call:
lm(formula = surprisal ~ definiteness * argPos * np_start_idx, 
    data = df_nps)

Residuals:
     Min       1Q   Median       3Q      Max 
-12.8160  -2.4721  -0.2779   2.1431  18.4841 

Coefficients:
                                       Estimate Std. Error t value Pr(>|t|)    
(Intercept)                             15.3990     0.3112  49.483  < 2e-16 ***
definitenessdef                         -0.7413     0.4562  -1.625  0.10430    
argPossbj                                1.6067     0.3665   4.383  1.2e-05 ***
np_start_idx                            -0.1664     0.0900  -1.849  0.06461 .  
definitenessdef:argPossbj                1.5125     0.5076   2.980  0.00291 ** 
definitenessdef:np_start_idx             0.1775     0.1341   1.324  0.18554    
argPossbj:np_start_idx                   0.1097     0.3859   0.284  0.77629    
definitenessdef:argPossbj:np_start_idx  -0.5582     0.4317  -1.293  0.19608    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Res

In [ ]:
mod_token_index <- lm(data = df_sentences, log(Word_Surprisal) ~ Word_Token_Index)

summary(mod_token_index)


Call:
lm(formula = log(Word_Surprisal) ~ Word_Token_Index, data = df_sentences)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.0281 -0.1642 -0.0030  0.1782  1.2197 

Coefficients:
                   Estimate Std. Error t value Pr(>|t|)    
(Intercept)       2.7804634  0.0010075 2759.84   <2e-16 ***
Word_Token_Index -0.0105505  0.0003257  -32.39   <2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.2973 on 207929 degrees of freedom
  (60401 observations deleted due to missingness)
Multiple R-squared:  0.005022,	Adjusted R-squared:  0.005017 
F-statistic:  1049 on 1 and 207929 DF,  p-value: < 2.2e-16


In [ ]:
options(repr.plot.width = 12, repr.plot.height = 8)

plot <- ggplot(data = df, aes(x = argPos, y = surprisal, fill = definiteness)) +
    geom_boxplot(outlier.shape = NA) +
        coord_cartesian(ylim = c(0, 35)) +
    labs(
        title = "Argument Position, Definiteness, and Surprisal",
        x = "Argument Position",
        y = "Surprisal",
        fill = "Definiteness"
    ) + 
    scale_x_discrete(labels = c("sbj" = "Subject", "obj" = "Object")) +
    scale_fill_discrete(labels = c("def" = "Definite", "indef" = "Indefinite"))+
    plotFont(3) 


plot

In [6]:
df <- open_dataset("Old or lost files/Results 12-9 5PM/12-9_5PM_Full-Data_UNPROCESSED.parquet") %>% collect()

In [7]:
head(df)

file_name,Sentence_ID,Filename,Modality,Sentence_Text,Sent_Verb_Count,Sent_Auxiliary_Count,Sent_Subject_Count,Sent_Tot_Obj_Count,Sent_Dir_Object_Count,⋯,NP_Head_Dependency,NP_Det_Dependency,NP_Head_Text,NP_Det_Text,NP_Sum_Surprisal,NP_Mean_Surprisal,NP_Argument,NP_Number,NP_Definiteness,Is_Head_Noun
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<lgl>
D:/BNC Full Data/12-9_5PM Run/CSV/A00.csv,A00_0001,A00.xml,written,is a condition caused by a virus called HIV ( Human Immuno Deficiency Virus ) .,2,1,0,0,0,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
D:/BNC Full Data/12-9_5PM Run/CSV/A00.csv,A00_0001,A00.xml,written,is a condition caused by a virus called HIV ( Human Immuno Deficiency Virus ) .,2,1,0,0,0,⋯,attr,det,condition,a,25.90625,12.95312,non-arg,singular,indefinite,FALSE
D:/BNC Full Data/12-9_5PM Run/CSV/A00.csv,A00_0001,A00.xml,written,is a condition caused by a virus called HIV ( Human Immuno Deficiency Virus ) .,2,1,0,0,0,⋯,attr,det,condition,a,25.90625,12.95312,non-arg,singular,indefinite,TRUE
D:/BNC Full Data/12-9_5PM Run/CSV/A00.csv,A00_0001,A00.xml,written,is a condition caused by a virus called HIV ( Human Immuno Deficiency Virus ) .,2,1,0,0,0,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
D:/BNC Full Data/12-9_5PM Run/CSV/A00.csv,A00_0001,A00.xml,written,is a condition caused by a virus called HIV ( Human Immuno Deficiency Virus ) .,2,1,0,0,0,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
D:/BNC Full Data/12-9_5PM Run/CSV/A00.csv,A00_0001,A00.xml,written,is a condition caused by a virus called HIV ( Human Immuno Deficiency Virus ) .,2,1,0,0,0,⋯,pobj,det,virus,a,28.56250,14.28125,prep_object,singular,indefinite,FALSE


In [ ]:
df %>% 
    count(Filename, sort = TRUE)

Filename,n
<chr>,<int>
HHV.xml,471934
HHX.xml,443991
K97.xml,391894
HHW.xml,355598
CRM.xml,329289
HH3.xml,323448
K5D.xml,319256
K5M.xml,290143
CBF.xml,275478


: 